In [2]:
import os
os.chdir('mmsegmentation')

In [3]:
os.getcwd()

'/home/featurize/work/mmsegmentation_25717_mine/mmsegmentation'

In [4]:
import os
import numpy as np
import cv2
from tqdm import tqdm

from mmseg.apis import init_model, inference_model, show_result_pyplot
import mmcv

import matplotlib.pyplot as plt
%matplotlib inline

/environment/miniconda3/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/environment/miniconda3/lib/python3.10/site-packages/mmengine/optim/optimizer/zero_optimizer.py:11: DeprecationWarning: `TorchScript` support for functional optimizers is deprecated and will be removed in a future PyTorch release. Consider using the `torch.compile` optimizer instead.
  from torch.distributed.optim import \


In [5]:
# 模型 config 配置文件
config_file = '/home/featurize/work/mmsegmentation_25717_mine/mmsegmentation/work_dirs/dinov3_swin_upernet_swinmainE/dinov3_swinV1-Copy1.py'

# 模型 checkpoint 权重文件
checkpoint_file = '/home/featurize/work/mmsegmentation_25717_mine/mmsegmentation/work_dirs/dinov3_swin_upernet_swinmainE/epoch_89-Copy1.pth'

# 计算硬件
# device = 'cpu'
device = 'cuda:0'

In [6]:
# # 模型 config 配置文件
# config_file = '/home/featurize/work/mmsegmentation_25717_mine/mmsegmentation/work_dirs/Hu-UNet/unet.py'

# # 模型 checkpoint 权重文件
# checkpoint_file = '/home/featurize/work/mmsegmentation_25717_mine/mmsegmentation/work_dirs/Hu-UNet/best_mIoU_iter_40000.pth'

# # 计算硬件
# # device = 'cpu'
# device = 'cuda:0'

In [7]:
model = init_model(config_file, checkpoint_file, device=device)

/home/featurize/work/mmsegmentation_25717_mine/mmsegmentation/mmseg/models/builder.py:36: UserWarning: ``build_loss`` would be deprecated soon, please use ``mmseg.registry.MODELS.build()`` 
  warnings.warn('``build_loss`` would be deprecated soon, please use '


Loads checkpoint by local backend from path: /home/featurize/work/mmsegmentation_25717_mine/mmsegmentation/work_dirs/dinov3_swin_upernet_swinmainE/epoch_89-Copy1.pth


/environment/miniconda3/lib/python3.10/site-packages/mmengine/runner/checkpoint.py:347: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(filename, map_l

In [8]:
!pip install /home/featurize/work/mmsegmentation_25717_mine/mmsegmentation/geoai_GDAL-3.4.3-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl

Looking in indexes: https://pypi.tuna.tsinghua.edu.cn/simple
Processing ./geoai_GDAL-3.4.3-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl
geoai-GDAL is already installed with the same version as the provided wheel. Use --force-reinstall to force an installation of the wheel.


In [9]:
# 每个类别的 BGR 配色
palette = [
    ['grassland', [127,127,127]],
    ['forest', [0,0,200]],
    ['building', [0,200,0]],
    ['road', [144,238,144]],
    ['bareground', [30,30,30]],
    ['water', [8,189,251]]
]

palette_dict = {}
for idx, each in enumerate(palette):
    palette_dict[idx] = each[1]

In [10]:
palette_dict

{0: [127, 127, 127],
 1: [0, 0, 200],
 2: [0, 200, 0],
 3: [144, 238, 144],
 4: [30, 30, 30],
 5: [8, 189, 251]}

In [11]:
# PATH_IMAGE = '/home/featurize/data/1/'

# PATH_IMAGE = '/home/featurize/data/yunnan_dataset/img_dir/val/'

PATH_IMAGE = '/home/featurize/data/jiangxi_dataset1204/jiangxi_dataset/img_dir'

In [12]:
os.chdir(PATH_IMAGE)

In [13]:
opacity=0.1
def process_single_img(img_path, save=False):
    
    img_bgr = cv2.imread(img_path)

    # 语义分割预测
    result = inference_model(model, img_bgr)
    pred_mask = result.pred_sem_seg.data[0].cpu().numpy()

    # 将预测的整数ID，映射为对应类别的颜色
    pred_mask_bgr = np.zeros((pred_mask.shape[0], pred_mask.shape[1], 3))
    for idx in palette_dict.keys():
        pred_mask_bgr[np.where(pred_mask==idx)] = palette_dict[idx]
    pred_mask_bgr = pred_mask_bgr.astype('uint8')

    # 将语义分割预测图和原图叠加显示
    pred_viz = cv2.addWeighted(img_bgr, opacity, pred_mask_bgr, 1-opacity, 0)
    
    # 保存图像至 outputs/testset-pred 目录
    if save:
        save_path = os.path.join('/home/featurize/work/mmsegmentation_25717_mine/mmsegmentation/outputs/testset-pred', 'pred-'+img_path.split('/')[-1])
        cv2.imwrite(save_path, pred_viz)

In [14]:
# # 模型 config 配置文件
# config_file = '/home/featurize/work/mmsegmentation_25717_mine/mmsegmentation/work_dirs/Hu-UNet/unet.py'

# # 模型 checkpoint 权重文件
# checkpoint_file = '/home/featurize/work/mmsegmentation_25717_mine/mmsegmentation/work_dirs/Hu-UNet/best_mIoU_iter_40000.pth'
# # 计算硬件
# # device = 'cpu'
# device = 'cuda:0'

In [15]:
# 模型 config 配置文件
config_file = '/home/featurize/work/mmsegmentation_25717_mine/mmsegmentation/work_dirs/dinov3_swin_upernet_swinmainE/dinov3_swinV1-Copy1.py'

# 模型 checkpoint 权重文件
checkpoint_file = '/home/featurize/work/mmsegmentation_25717_mine/mmsegmentation/work_dirs/dinov3_swin_upernet_swinmainE/epoch_89-Copy1.pth'

# 计算硬件
# device = 'cpu'
device = 'cuda:0'

In [16]:
model = init_model(config_file, checkpoint_file, device=device)

Loads checkpoint by local backend from path: /home/featurize/work/mmsegmentation_25717_mine/mmsegmentation/work_dirs/dinov3_swin_upernet_swinmainE/epoch_89-Copy1.pth


In [17]:
from osgeo import gdal
import numpy as np
from typing import Optional


def load_rs_image_with_gdal(img_path: str, to_float32: bool = True) -> Optional[np.ndarray]:
    """
    基于 GDAL 加载遥感图像，功能类似 cv2.imread，但保留遥感图像多波段支持和原始波段顺序
    （逻辑完全来自 LoadSingleRSImageFromFile 的 transform 方法）
    
    Args:
        img_path: 遥感图像路径（如 .tif、.png 等 GDAL 支持格式）
        to_float32: 是否将图像数据转为 float32（默认 True，与 LoadSingleRSImageFromFile 一致）
    
    Returns:
        图像数组（shape: [H, W, C]，C 为波段数）；加载失败返回 None（类似 cv2.imread 返回 None）
    """
    # 检查 GDAL 是否可用（与 LoadSingleRSImageFromFile 一致）
    if gdal is None:
        raise RuntimeError("gdal is not installed, cannot load remote sensing image")
    
    # 1. 用 GDAL 打开图像文件（类似 cv2.imread 打开文件）
    ds = gdal.Open(img_path)
    if ds is None:  # 打开失败（如路径错误、文件损坏），返回 None（模仿 cv2.imread 行为）
        print(f"Warning: Failed to open image with GDAL: {img_path}")
        return None
    
    # 2. 读取图像数组并调整维度顺序（关键：GDAL 默认输出 [C, H, W]，转成 [H, W, C]，与 cv2 一致）
    # ds.ReadAsArray() → 输出 shape: (通道数C, 高度H, 宽度W)
    # np.einsum('ijk->jki') → 维度重排为 (H, W, C)
    img_array = np.einsum('ijk->jki', ds.ReadAsArray())
    
    # 3. 数据类型转换（与 LoadSingleRSImageFromFile 一致，默认转 float32）
    if to_float32:
        img_array = img_array.astype(np.float32)
    
    # 4. 关闭 GDAL 数据集（释放资源）
    ds = None
    
    return img_array

In [18]:
def process_single_img(img_path, model, save=False):
    # 1. 用 GDAL 加载图像（替代 cv2.imread(img_path)）
    img_array = load_rs_image_with_gdal(img_path, to_float32=True)
    if img_array is None:  # 加载失败，跳过
        return
    
    # 2. 关键：将加载的数组传给 inference_model（此时流水线用 LoadImageFromNDArray）
    try:
        # inference_model 内部会调用 _preprare_data，自动识别数组并构建 results['img']
        result = inference_model(model, img_array)
    except Exception as e:
        print(f"Inference error for {img_path}: {str(e)}")
        return
    
    # 3. 提取单波段预测掩码（后续逻辑不变）
    pred_mask = result.pred_sem_seg.data[0].cpu().numpy().astype('uint8')
    
    # 4. 保存预测图（后续逻辑不变）
    if save:
        output_dir = '/home/featurize/work/mmsegmentation_25717_mine/mmsegmentation/outputs/testset-pred'
        os.makedirs(output_dir, exist_ok=True)
        save_filename = 'pred-' + os.path.basename(img_path)
        save_path = os.path.join(output_dir, save_filename)
        cv2.imwrite(save_path, pred_mask)  # 用 cv2 保存单波段掩码（不影响）


# 批量处理示例（确保传入完整路径）
import os
from tqdm import tqdm

# img_dir = "/home/featurize/data/yunnan_dataset/img_dir/val" # 你的图像目录
img_dir = "/home/featurize/data/jiangxi_dataset1204/jiangxi_dataset/img_dir"
for img_filename in tqdm(os.listdir(img_dir), desc="Processing RS Images"):
    full_img_path = os.path.join(img_dir, img_filename)  # 完整路径
    process_single_img(full_img_path, model, save=True)

Processing RS Images: 100%|██████████| 262/262 [00:38<00:00,  6.72it/s]


In [19]:
import os
import cv2
import numpy as np
from tqdm import tqdm
from osgeo import gdal  # 确保GDAL已安装


# ------------------------------
# 1. 修改：加载图像时同时获取投影和地理变换
# ------------------------------
def load_rs_image_with_gdal(img_path: str, to_float32: bool = True) -> tuple:
    """
    基于GDAL加载遥感图像，返回（图像数组, 投影信息, 地理变换参数）
    替代原仅返回数组的逻辑，新增投影/地理变换用于后续TIFF保存
    """
    if gdal is None:
        raise RuntimeError("GDAL未安装，无法加载遥感图像和投影信息")
    
    # 用GDAL打开图像文件
    ds = gdal.Open(img_path, gdal.GA_ReadOnly)
    if ds is None:
        print(f"警告：无法打开图像 {img_path}（路径错误或文件损坏）")
        return (None, None, None)  # 数组+投影+变换均返回None，标记加载失败
    
    # 关键：读取原始图像的投影和地理变换（保留投影的核心数据）
    proj = ds.GetProjection()  # 投影字符串（如WGS84、UTM）
    geotrans = ds.GetGeoTransform()  # 地理变换参数（左上角坐标、像素分辨率等）
    
    # 读取图像数组并调整维度（GDAL默认[C, H, W] → 转为[H, W, C]，与cv2一致）
    img_array = np.einsum('ijk->jki', ds.ReadAsArray())
    
    # 数据类型转换（保持原逻辑）
    if to_float32:
        img_array = img_array.astype(np.float32)
    
    # 关闭GDAL数据集，释放资源
    ds = None
    
    return (img_array, proj, geotrans)  # 返回（数组, 投影, 地理变换）


# ------------------------------
# 2. 新增：用GDAL保存带投影的TIFF文件
# ------------------------------
def save_tiff_with_projection(
    save_path: str,
    pred_mask: np.ndarray,
    proj: str,
    geotrans: tuple,
    dtype: int = gdal.GDT_Byte  # 掩码数据类型（uint8，适合类别ID）
) -> bool:
    """
    用GDAL保存预测掩码为TIFF文件，并注入原始图像的投影和地理变换
    返回：保存成功True，失败False
    """
    if gdal is None:
        print("GDAL未安装，无法保存带投影的TIFF")
        return False
    
    # 检查输入掩码维度（必须是单波段[H, W]，避免多波段错误）
    if pred_mask.ndim != 2:
        print(f"错误：预测掩码维度为{pred_mask.ndim}，需为单波段[H, W]")
        return False
    
    # 获取掩码的高度和宽度
    height, width = pred_mask.shape
    
    # 1. 创建GDAL TIFF驱动
    driver = gdal.GetDriverByName('GTiff')
    if driver is None:
        print("错误：无法创建GTiff驱动（GDAL配置异常）")
        return False
    
    # 2. 创建TIFF文件（1个波段，对应单波段掩码）
    # 参数：路径、宽度、高度、波段数、数据类型
    out_ds = driver.Create(save_path, width, height, 1, dtype)
    if out_ds is None:
        print(f"错误：无法创建TIFF文件 {save_path}（路径无权限或磁盘满）")
        return False
    
    # 3. 注入原始图像的投影和地理变换（核心：保留投影）
    out_ds.SetProjection(proj)
    out_ds.SetGeoTransform(geotrans)
    
    # 4. 写入预测掩码数据（波段索引从1开始，GDAL默认规则）
    out_band = out_ds.GetRasterBand(1)
    out_band.WriteArray(pred_mask)  # 写入[H, W]掩码
    
    # 5. 刷新缓存，确保数据写入磁盘
    out_band.FlushCache()
    out_ds.FlushCache()
    
    # 6. 释放资源
    out_band = None
    out_ds = None
    
    print(f"成功保存带投影的TIFF：{save_path}")
    return True


# ------------------------------
# 3. 修改：原process_single_img函数，添加TIFF保存逻辑
# ------------------------------
def process_single_img(img_path, model, save=False):
    # 1. 用修改后的GDAL加载：获取图像数组+投影+地理变换
    img_array, original_proj, original_geotrans = load_rs_image_with_gdal(img_path, to_float32=True)
    if img_array is None:  # 加载失败（数组为None），跳过
        return
    
    # 2. 语义分割推理（保持原逻辑不变）
    try:
        result = inference_model(model, img_array)
    except Exception as e:
        print(f"推理错误 {img_path}：{str(e)}")
        return
    
    # 3. 提取单波段预测掩码（保持原逻辑不变）
    pred_mask = result.pred_sem_seg.data[0].cpu().numpy().astype('uint8')
    # 确保掩码是[H, W]单波段（若模型输出有多余维度，挤压掉）
    pred_mask = np.squeeze(pred_mask)
    
    # 4. 保存：新增TIFF（带投影）+ 保留原cv2保存（可选）
    if save:
        output_dir = '/home/featurize/work/mmsegmentation_25717_mine/mmsegmentation/outputs/testset-pred'
        os.makedirs(output_dir, exist_ok=True)  # 确保目录存在
        img_basename = os.path.basename(img_path)  # 原文件名（如1002.tif）
        
        # ------------------------------
        # 新增：保存带投影的TIFF
        # ------------------------------
        # 构建TIFF保存路径（前缀pred-，后缀.tif，覆盖原后缀）
        tiff_save_name = f"pred-{os.path.splitext(img_basename)[0]}.tif"
        tiff_save_path = os.path.join(output_dir, tiff_save_name)
        # 调用新增函数保存TIFF（传入原始投影和地理变换）
        save_tiff_with_projection(
            save_path=tiff_save_path,
            pred_mask=pred_mask,
            proj=original_proj,
            geotrans=original_geotrans
        )
        
        # ------------------------------
        # 保留：原cv2保存（如PNG，可选，不影响TIFF）
        # ------------------------------
        # cv2_save_path = os.path.join(output_dir, f"pred-{img_basename}")
        # cv2.imwrite(cv2_save_path, pred_mask)
        # print(f"成功保存cv2格式：{cv2_save_path}")


# ------------------------------
# 批量处理示例（保持原逻辑不变）
# ------------------------------
if __name__ == "__main__":
    # 假设model已通过init_model加载完成（示例代码）
    # from mmseg.apis import init_model
    # config_path = "你的配置文件路径.py"
    # checkpoint_path = "你的权重文件路径.pth"
    # model = init_model(config_path, checkpoint_path, device='cuda:0')
    # model.eval()
    
    img_dir = "/home/featurize/data/jiangxi_dataset1204/jiangxi_dataset/img_dir"  # 你的图像目录
    # img_dir = "/home/featurize/data/yunnan_dataset/img_dir/val"  # 你的图像目录
    for img_filename in tqdm(os.listdir(img_dir), desc="Processing RS Images"):
        full_img_path = os.path.join(img_dir, img_filename)  # 完整路径
        process_single_img(full_img_path, model, save=True)  # 调用修改后的函数

Processing RS Images:   0%|          | 0/262 [00:00<?, ?it/s]Warning 1: PROJ: proj_create_from_database: Cannot find proj.db
ERROR 1: PROJ: proj_create_from_name: Cannot find proj.db
Processing RS Images:   0%|          | 1/262 [00:00<00:32,  7.97it/s]Warning 1: PROJ: proj_create_from_database: Cannot find proj.db
ERROR 1: PROJ: proj_create_from_name: Cannot find proj.db
Processing RS Images:   1%|          | 2/262 [00:00<00:31,  8.15it/s]Warning 1: PROJ: proj_create_from_database: Cannot find proj.db


成功保存带投影的TIFF：/home/featurize/work/mmsegmentation_25717_mine/mmsegmentation/outputs/testset-pred/pred-72.tif
成功保存带投影的TIFF：/home/featurize/work/mmsegmentation_25717_mine/mmsegmentation/outputs/testset-pred/pred-148.tif


ERROR 1: PROJ: proj_create_from_name: Cannot find proj.db
Processing RS Images:   1%|          | 3/262 [00:00<00:31,  8.12it/s]Warning 1: PROJ: proj_create_from_database: Cannot find proj.db
ERROR 1: PROJ: proj_create_from_name: Cannot find proj.db
Processing RS Images:   2%|▏         | 4/262 [00:00<00:31,  8.11it/s]Warning 1: PROJ: proj_create_from_database: Cannot find proj.db


成功保存带投影的TIFF：/home/featurize/work/mmsegmentation_25717_mine/mmsegmentation/outputs/testset-pred/pred-241.tif
成功保存带投影的TIFF：/home/featurize/work/mmsegmentation_25717_mine/mmsegmentation/outputs/testset-pred/pred-135.tif


ERROR 1: PROJ: proj_create_from_name: Cannot find proj.db
Processing RS Images:   2%|▏         | 5/262 [00:00<00:31,  8.15it/s]Warning 1: PROJ: proj_create_from_database: Cannot find proj.db
ERROR 1: PROJ: proj_create_from_name: Cannot find proj.db
Processing RS Images:   2%|▏         | 6/262 [00:00<00:31,  8.20it/s]Warning 1: PROJ: proj_create_from_database: Cannot find proj.db


...

成功保存带投影的TIFF：/home/featurize/work/mmsegmentation_25717_mine/mmsegmentation/outputs/testset-pred/pred-68.tif
成功保存带投影的TIFF：/home/featurize/work/mmsegmentation_25717_mine/mmsegmentation/outputs/testset-pred/pred-118.tif


ERROR 1: PROJ: proj_create_from_name: Cannot find proj.db
Processing RS Images:  99%|█████████▉| 259/262 [00:32<00:00,  8.27it/s]Warning 1: PROJ: proj_create_from_database: Cannot find proj.db
ERROR 1: PROJ: proj_create_from_name: Cannot find proj.db
Processing RS Images:  99%|█████████▉| 260/262 [00:32<00:00,  8.30it/s]Warning 1: PROJ: proj_create_from_database: Cannot find proj.db


成功保存带投影的TIFF：/home/featurize/work/mmsegmentation_25717_mine/mmsegmentation/outputs/testset-pred/pred-9.tif
成功保存带投影的TIFF：/home/featurize/work/mmsegmentation_25717_mine/mmsegmentation/outputs/testset-pred/pred-37.tif


ERROR 1: PROJ: proj_create_from_name: Cannot find proj.db
Processing RS Images: 100%|█████████▉| 261/262 [00:32<00:00,  8.32it/s]Warning 1: PROJ: proj_create_from_database: Cannot find proj.db
ERROR 1: PROJ: proj_create_from_name: Cannot find proj.db
Processing RS Images: 100%|██████████| 262/262 [00:32<00:00,  8.09it/s]

成功保存带投影的TIFF：/home/featurize/work/mmsegmentation_25717_mine/mmsegmentation/outputs/testset-pred/pred-91.tif
成功保存带投影的TIFF：/home/featurize/work/mmsegmentation_25717_mine/mmsegmentation/outputs/testset-pred/pred-209.tif
